In [ ]:
import os
import arcpy
import pyspark.sql.functions as F

from typing import Tuple
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType
from spark_esri import spark_start, spark_stop

In [ ]:
extra_java_options = "-XX:+UseCompressedOops -XX:+AggressiveHeap"

config = {
    "spark.driver.memory": "31G",
    "spark.executor.memory": "31G",
    "spark.driver.extraJavaOptions": extra_java_options,
    "spark.executor.extraJavaOptions": extra_java_options,
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.kryo.unsafe": True,
    "spark.memory.offHeap.enable": True,
    "spark.memory.offHeap.size": "31G",
}

spark_stop()
spark = spark_start(config=config)

In [ ]:
sp_ref = arcpy.SpatialReference(32639)
fields = ["SHAPE@WKB","classname","classvalue","ImageURI"]

In [ ]:
with arcpy.da.SearchCursor("Slicks39N", fields, spatial_reference=sp_ref) as rows:
    df = spark\
        .createDataFrame(rows, "WKB binary,Classname string,Classvalue int,ImageURI string")\
        .cache()

In [ ]:
return_type = StructType([
    StructField("pid", LongType(), False),
    StructField("len", IntegerType(), False)
])

def create_fc(pid:int) -> str:
    with arcpy.EnvManager(overwriteOutput=True):
        ws = "memory"
        nm = f"f{pid}"
        fc = os.path.join(ws, nm)
        arcpy.management.Delete(fc)
        sp_ref = arcpy.SpatialReference(32639)
        arcpy.management.CreateFeatureclass(ws, nm, "POLYGON", spatial_reference=sp_ref)
        arcpy.management.AddFields(fc, [
            ("Classname", "Text"),
            ("Classvalue", "Long"),
            ("ImageURI", "Text")
        ])
        return fc

@F.udf(returnType=return_type)
def etd4dl(pid:int,col:list) -> Tuple[int,int]:
    create_fc(pid)
    return pid,len(col)

In [ ]:
create_fc(3)

In [ ]:
df\
    .repartition(4)\
    .withColumn("pid",F.spark_partition_id())\
    .groupBy("pid")\
    .agg(F.collect_list(F.struct("WKB","Classname","Classvalue","ImageURI")).alias("col"))\
    .select(etd4dl("pid","col").alias("etd4dl"))\
    .show()